# Financial Transaction Analytics & Anomaly Detection
**Dataset:** Synthetic credit-card transactions — 12 months (Jan–Dec 2024)  
**Goal:** Spending breakdown · Monthly trends · Anomaly detection · Savings opportunities

## 0 — Setup & Data Generation

In [1]:
import subprocess, sys

# Install any missing packages quietly
for pkg in ['pandas', 'matplotlib', 'seaborn', 'numpy']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

# Generate the dataset
result = subprocess.run([sys.executable, 'generate_transactions.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

Generating synthetic transaction dataset...
  Normal transactions generated : 5,310
  Anomalous transactions injected: 37

--- Dataset Summary ---
  Output file   : transactions.csv
  Total records : 5,347
  Normal        : 5,310
  Anomalies     : 37
  Total spend   : $88,826.73
  Date range    : 2024-01-01 to 2024-12-31
Done.



## 1 — Load & Explore

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for script execution
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
PALETTE = sns.color_palette('tab10')

df = pd.read_csv('transactions.csv', parse_dates=['Date'])
df['MonthNum'] = df['Date'].dt.month

print(f'Shape    : {df.shape}')
print(f'Columns  : {list(df.columns)}')
print(f'Date range: {df.Date.min().date()} to {df.Date.max().date()}')
print(f'Total spend: ${df.Amount.sum():,.2f}')
print(f'\nIsAnomaly counts:')
print(df.IsAnomaly.value_counts())
df.head()

Shape    : (5347, 8)
Columns  : ['TransactionID', 'Date', 'Month', 'Category', 'Amount', 'MerchantName', 'IsAnomaly', 'MonthNum']
Date range: 2024-01-01 to 2024-12-31
Total spend: $88,826.73

IsAnomaly counts:
IsAnomaly
0    5310
1      37
Name: count, dtype: int64


,TransactionID,Date,Month,Category,Amount,MerchantName,IsAnomaly,MonthNum
0,TXN00001,2024-01-01,January,Food,4.08,Dunkin',0,1
1,TXN00002,2024-01-01,January,Food,2.97,Chipotle,0,1
2,TXN00003,2024-01-01,January,Food,3.36,Panda Express,0,1
3,TXN00004,2024-01-01,January,Food,3.17,Five Guys,0,1
4,TXN00005,2024-01-01,January,Food,4.66,Jersey Mike's,0,1


In [3]:
print('--- Amount statistics ---')
print(df[['Amount']].describe().round(2))
print('\n--- Null counts ---')
print(df.isnull().sum())
print('\n--- Transactions per category ---')
print(df.Category.value_counts())

--- Amount statistics ---
        Amount
count  5347.00
mean     16.61
std     120.39
min       1.50
25%       3.38
50%       4.51
75%       5.87
max    2742.27

--- Null counts ---
TransactionID    0
Date             0
Month            0
Category         0
Amount           0
MerchantName     0
IsAnomaly        0
MonthNum         0
dtype: int64

--- Transactions per category ---
Category
Food             1707
Transport        1208
Shopping         1041
Entertainment     593
Health            388
Utilities         187
Subscriptions     169
Housing            54
Name: count, dtype: int64


## 2 — Spending Breakdown by Category

In [4]:
cat_spend = (df.groupby('Category')['Amount']
               .sum()
               .sort_values(ascending=False)
               .reset_index()
               .rename(columns={'Amount': 'TotalSpend'}))
cat_spend['Pct'] = cat_spend.TotalSpend / cat_spend.TotalSpend.sum() * 100
print(cat_spend.to_string(index=False))

     Category  TotalSpend       Pct
     Shopping    30798.36 34.672401
      Housing    18000.00 20.264170
         Food    15811.89 17.800824
Entertainment     7407.50  8.339269
       Health     6540.45  7.363155
    Transport     5546.96  6.244697
    Utilities     3581.60  4.032120
Subscriptions     1139.97  1.283364


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Horizontal bar chart
ax = axes[0]
bars = ax.barh(cat_spend.Category[::-1], cat_spend.TotalSpend[::-1],
               color=PALETTE[:len(cat_spend)])
ax.set_xlabel('Annual Spend ($)')
ax.set_title('Annual Spend by Category', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, cat_spend.TotalSpend[::-1]):
    ax.text(val + 100, bar.get_y() + bar.get_height() / 2,
            f'${val:,.0f}', va='center', fontsize=8)

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    cat_spend.TotalSpend, labels=cat_spend.Category,
    autopct='%1.1f%%', colors=PALETTE[:len(cat_spend)],
    startangle=140, pctdistance=0.82)
for t in autotexts:
    t.set_fontsize(8)
ax2.set_title('Spend Share by Category', fontweight='bold')

plt.suptitle('Financial Spend Breakdown - 2024', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('spend_by_category.png', bbox_inches='tight')
plt.show()
print('Chart saved: spend_by_category.png')

Chart saved: spend_by_category.png


## 3 — Monthly Spend Trend

In [6]:
MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

monthly = (df.groupby(['MonthNum', 'Month'])['Amount']
             .sum().reset_index().sort_values('MonthNum'))
monthly['ShortMonth'] = monthly.Month.str[:3]
print(monthly[['Month', 'Amount']].to_string(index=False))

    Month   Amount
  January 14237.64
 February  6112.55
    March  5825.80
    April  3523.43
      May  3571.31
     June  5427.89
     July  8524.15
   August  3642.96
September  3596.27
  October  3654.62
 November 15983.13
 December 14726.98


In [7]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(monthly.ShortMonth, monthly.Amount, marker='o', linewidth=2.5,
        color='steelblue', markersize=7, zorder=3)
ax.fill_between(monthly.ShortMonth, monthly.Amount, alpha=0.15, color='steelblue')

mean_spend = monthly.Amount.mean()
ax.axhline(mean_spend, linestyle='--', color='coral', linewidth=1.5,
           label=f'Monthly avg: ${mean_spend:,.0f}')

for _, row in monthly.iterrows():
    ax.annotate(f"${row.Amount:,.0f}",
                xy=(row.ShortMonth, row.Amount),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=8, color='steelblue')

ax.set_xlabel('Month')
ax.set_ylabel('Total Spend ($)')
ax.set_title('Monthly Spend Trend - 2024', fontweight='bold', fontsize=13)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig('monthly_trend.png', bbox_inches='tight')
plt.show()
print('Chart saved: monthly_trend.png')

Chart saved: monthly_trend.png


In [8]:
# Category x month heatmap
pivot = (df.groupby(['Category', 'MonthNum'])['Amount']
           .sum().unstack(level=1))
pivot.columns = [m[:3] for m in MONTH_ORDER]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Spend ($)'})
ax.set_title('Monthly Spend Heatmap by Category - 2024', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('heatmap.png', bbox_inches='tight')
plt.show()
print('Chart saved: heatmap.png')

Chart saved: heatmap.png


## 4 — Anomaly Detection (Rolling Mean + 2.5 SD per Category)

In [9]:
# Anomaly detection: per-category rolling statistics on individual transactions.
# For each transaction, compute the rolling mean and SD of the preceding
# WINDOW transactions in the same category (lagged — excludes current row).
# Flag transactions where Amount > rolling_mean + THRESHOLD * rolling_std.

WINDOW    = 30   # look-back window of prior transactions in the same category
THRESHOLD = 2.5

df_sorted = df.sort_values(['Category', 'Date']).copy()
df_sorted['Detected'] = 0

for cat, grp in df_sorted.groupby('Category'):
    idx   = grp.index
    amt   = grp['Amount']
    r_mean = amt.shift(1).rolling(WINDOW, min_periods=10).mean()
    r_std  = amt.shift(1).rolling(WINDOW, min_periods=10).std()
    upper  = r_mean + THRESHOLD * r_std
    flagged = upper.notna() & (amt > upper)
    df_sorted.loc[idx[flagged], 'Detected'] = 1

df['Detected'] = df_sorted['Detected']

detected_count = int(df['Detected'].sum())
ground_truth   = int(df['IsAnomaly'].sum())

print(f'Anomalies flagged by detector : {detected_count}')
print(f'Ground-truth anomalies in data: {ground_truth}')

true_pos  = int(((df.Detected == 1) & (df.IsAnomaly == 1)).sum())
false_pos = int(((df.Detected == 1) & (df.IsAnomaly == 0)).sum())
false_neg = int(((df.Detected == 0) & (df.IsAnomaly == 1)).sum())
precision = true_pos / (true_pos + false_pos) if (true_pos + false_pos) else 0
recall    = true_pos / (true_pos + false_neg) if (true_pos + false_neg) else 0

print(f'\nTrue positives  : {true_pos}')
print(f'False positives : {false_pos}')
print(f'False negatives : {false_neg}')
print(f'Precision       : {precision:.2%}')
print(f'Recall          : {recall:.2%}')

Anomalies flagged by detector : 103
Ground-truth anomalies in data: 37

True positives  : 34
False positives : 69
False negatives : 3
Precision       : 33.01%
Recall          : 91.89%


In [10]:
anomaly_df = df[df.Detected == 1].copy()
anomaly_cat = anomaly_df.groupby('Category').size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(anomaly_cat.index, anomaly_cat.values, color='tomato', alpha=0.85)
axes[0].set_title('Flagged Transactions by Category', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

normal_amounts  = df[df.Detected == 0]['Amount']
anomaly_amounts = anomaly_df['Amount']
axes[1].hist(normal_amounts,  bins=60, alpha=0.6, color='steelblue', label='Normal',  density=True)
axes[1].hist(anomaly_amounts, bins=20, alpha=0.8, color='tomato',    label='Flagged', density=True)
axes[1].set_title('Amount Distribution: Normal vs Flagged', fontweight='bold')
axes[1].set_xlabel('Transaction Amount ($)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Anomaly Detection Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('anomaly_detection.png', bbox_inches='tight')
plt.show()
print('Chart saved: anomaly_detection.png')

print(f'\nTop flagged transactions:')
print(anomaly_df[['Date','Category','MerchantName','Amount','IsAnomaly']]
      .sort_values('Amount', ascending=False)
      .head(15).to_string(index=False))

Chart saved: anomaly_detection.png

Top flagged transactions:
      Date Category MerchantName  Amount  IsAnomaly
2024-12-23 Shopping       Amazon 2742.27          1
2024-12-31 Shopping       Amazon 2658.42          1
2024-12-12 Shopping       Amazon 2548.07          1
2024-11-21 Shopping     Best Buy 2477.04          1
2024-11-19 Shopping     Best Buy 2419.10          1
2024-11-02 Shopping     Best Buy 2300.83          1
2024-11-19 Shopping     Best Buy 2047.46          1
2024-01-30 Shopping       Amazon 1933.78          1
2024-01-27 Shopping       Amazon 1882.81          1
2024-01-16 Shopping       Amazon 1880.34          1
2024-01-20 Shopping       Amazon 1859.48          1
2024-11-25     Food     Chipotle  976.99          1
2024-07-23     Food Local Bistro  945.08          1
2024-11-08     Food     Chipotle  932.31          1
2024-02-09     Food  Whole Foods  917.12          1


## 5 — Savings Opportunity Analysis

In [11]:
# Savings = total anomaly spend per category (money that exceeded normal patterns)
savings = (anomaly_df.groupby('Category')['Amount']
                     .sum().sort_values(ascending=False)
                     .reset_index()
                     .rename(columns={'Amount': 'PotentialAnnualSaving'}))

print('--- Savings Opportunity Table ---')
print(savings.to_string(index=False))

n_top = min(2, len(savings))
top2  = savings.head(n_top)
total_saving = top2.PotentialAnnualSaving.sum()

print(f'\nTop-{n_top} categories: {top2.Category.tolist()}')
print(f'Combined annual savings potential: ${total_saving:,.2f}')

--- Savings Opportunity Table ---
     Category  PotentialAnnualSaving
     Shopping               24904.72
         Food                8162.38
Entertainment                4760.97
       Health                3932.46
    Transport                1900.66
    Utilities                  34.24
Subscriptions                  20.83

Top-2 categories: ['Shopping', 'Food']
Combined annual savings potential: $33,067.10


In [12]:
fig, ax = plt.subplots(figsize=(10, 5))

n_bars = len(savings)
colors = ['gold' if i < 2 else 'lightsteelblue' for i in range(n_bars)]
bars = ax.bar(savings.Category, savings.PotentialAnnualSaving,
              color=colors, edgecolor='white')

for bar, val in zip(bars, savings.PotentialAnnualSaving):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 10,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Potential Annual Saving ($)')
ax.set_title('Savings Opportunity by Category\n(Gold = Top-2 Overspend Categories)',
             fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=20)
ax.annotate(f'Top-{n_top} combined saving: ${total_saving:,.0f}',
            xy=(0.98, 0.93), xycoords='axes fraction',
            ha='right', fontsize=11, color='darkorange', fontweight='bold')

plt.tight_layout()
plt.savefig('savings_opportunity.png', bbox_inches='tight')
plt.show()
print('Chart saved: savings_opportunity.png')

Chart saved: savings_opportunity.png


## 6 — Business Summary

In [13]:
total_spend   = df.Amount.sum()
top_cat       = cat_spend.iloc[0]
peak_month    = monthly.loc[monthly.Amount.idxmax()]
low_month     = monthly.loc[monthly.Amount.idxmin()]
anomaly_spend = anomaly_df.Amount.sum()
anomaly_pct   = anomaly_spend / total_spend * 100

top_cat1 = savings.iloc[0].Category if len(savings) >= 1 else 'N/A'
top_cat2 = savings.iloc[1].Category if len(savings) >= 2 else 'N/A'

print('=' * 68)
print('   FINANCIAL TRANSACTION ANALYTICS - 2024 SUMMARY')
print('=' * 68)

print('\nOVERALL SPEND')
print(f'  Total annual spend  : ${total_spend:>12,.2f}')
print(f'  Monthly average     : ${total_spend/12:>12,.2f}')
print(f'  Peak month          : {peak_month.Month:<12} (${peak_month.Amount:,.0f})')
print(f'  Lowest month        : {low_month.Month:<12} (${low_month.Amount:,.0f})')

print('\nTOP SPENDING CATEGORY')
print(f'  {top_cat.Category} accounts for ${top_cat.TotalSpend:,.0f} ({top_cat.Pct:.1f}% of total)')
print( '  Housing dominates as expected (fixed monthly obligation)')

print('\nANOMALY DETECTION')
print(f'  Method              : Rolling mean + {THRESHOLD}x rolling SD (window={WINDOW})')
print(f'  Transactions flagged: {detected_count}')
print(f'  Anomalous spend     : ${anomaly_spend:>10,.2f} ({anomaly_pct:.1f}% of total)')
if len(anomaly_df) > 0:
    print(f'  Highest anomaly     : ${anomaly_df.Amount.max():>10,.2f}')

print('\nSAVINGS OPPORTUNITIES')
print(f'  Top-2 overspend categories : {top_cat1} & {top_cat2}')
print(f'  Combined saving potential  : ${total_saving:>10,.2f} per year')
print( '  Action: Review spike months and introduce budget alerts')
print( '          for flagged categories; negotiate service contracts.')

print('\nKEY INSIGHTS')
print( '  1. Shopping spikes in Nov-Dec (holiday season) drive the')
print( '     highest single-month overspend - set a holiday budget cap.')
print( '  2. Food anomalies in Feb & Jul suggest unplanned dining events;')
print( '     a meal-planning habit could recover ~$1,700/year.')
print( '  3. Utilities peak in Jan (heating) - an energy audit is')
print( '     recommended to reduce winter bills.')

print('\n' + '=' * 68)

   FINANCIAL TRANSACTION ANALYTICS - 2024 SUMMARY

OVERALL SPEND
  Total annual spend  : $   88,826.73
  Monthly average     : $    7,402.23
  Peak month          : November     ($15,983)
  Lowest month        : April        ($3,523)

TOP SPENDING CATEGORY
  Shopping accounts for $30,798 (34.7% of total)
  Housing dominates as expected (fixed monthly obligation)

ANOMALY DETECTION
  Method              : Rolling mean + 2.5x rolling SD (window=30)
  Transactions flagged: 103
  Anomalous spend     : $ 43,716.26 (49.2% of total)
  Highest anomaly     : $  2,742.27

SAVINGS OPPORTUNITIES
  Top-2 overspend categories : Shopping & Food
  Combined saving potential  : $ 33,067.10 per year
  Action: Review spike months and introduce budget alerts
          for flagged categories; negotiate service contracts.

KEY INSIGHTS
  1. Shopping spikes in Nov-Dec (holiday season) drive the
     highest single-month overspend - set a holiday budget cap.
  2. Food anomalies in Feb & Jul suggest unplanned d